# Search demo: BM25, pretrained and fine-tuned embeddings, hybrid

Top results of four systems for the same query, on BEIR SciFact (5,183 abstracts). Validation queries show which results are judged relevant (✓); free-text queries have no judgements. Run from the repository root; the notebook needs `data/` (corpus, embeddings, test scores) and `models/` (fine-tuned MiniLM), which the scripts in `src/` produce and git does not track. Numbers and method are in [results.md](results.md).

In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer

from src.data import load_corpus, load_qrels, load_queries
from src.dense import DenseSearch, encode_corpus
from src.evaluate import bootstrap_ci
from src.hybrid import Hybrid
from src.keyword import build_bm25
from src.score_dense import BGE_PREFIX
from src.splits import load_splits
from src.train import MODELS_DIR

pd.set_option("display.max_colwidth", 60, "display.width", 300)

BGE = "BAAI/bge-base-en-v1.5"
FT_RUN, FT_EPOCH = "all-MiniLM-L6-v2_hard_s0", 3

corpus, queries, qrels = load_corpus(), load_queries(), load_qrels("train")
val = load_splits().val

bm25 = build_bm25(corpus)
bge = SentenceTransformer(BGE, device="cpu")
doc_ids, bge_emb, _ = encode_corpus(bge, corpus, BGE)
bge_search = DenseSearch(bge, doc_ids, bge_emb, BGE_PREFIX)
tuned = SentenceTransformer(str(MODELS_DIR / FT_RUN), device="cpu")
_, tuned_emb, _ = encode_corpus(tuned, corpus, f"ft_{FT_RUN}_epoch{FT_EPOCH}")
SYSTEMS = {
    "BM25 (stemmed)": bm25.system(),
    "bge-base": bge_search.system(),
    "hybrid BM25 + bge-base": Hybrid(bm25, bge_search, doc_ids, "wsum", 0.2).system(),
    "MiniLM fine-tuned": DenseSearch(tuned, doc_ids, tuned_emb).system(),
}

In [2]:
def show(query, relevant=(), k=5):
    columns = {}
    for name, system in SYSTEMS.items():
        top = system(query)[:k]
        columns[name] = [("✓ " if d in relevant else "  ") + corpus[d]["title"] for d in top]
    return pd.DataFrame(columns, index=range(1, k + 1))


def show_validation(qid, k=5):
    print(queries[qid])
    print("relevant:", " | ".join(corpus[d]["title"] for d in qrels[qid]))
    return show(queries[qid], set(qrels[qid]), k)

## Different words for the same thing

The claim says "students"; the relevant paper says "pupils". The embedding models rank it first; BM25 ranks it fifth, behind papers that match other words.

In [3]:
show_validation("187")

British male students are bullied more than British female students.
relevant: Bullying in school: are short pupils at risk? Questionnaire study in a cohort.


,BM25 (stemmed),bge-base,hybrid BM25 + bge-base,MiniLM fine-tuned
1,Predictors of Attrition and Academic Success of Medica...,✓ Bullying in school: are short pupils at risk? Question...,✓ Bullying in school: are short pupils at risk? Question...,✓ Bullying in school: are short pupils at risk? Question...
2,Inheritance of coronary artery disease in men: an anal...,What do medical schools teach about women's health and...,Predictors of Attrition and Academic Success of Medica...,Patterns of chlamydia testing in different settings an...
3,Computer assisted learning in undergraduate medical ed...,Risk factors at medical school for subsequent professi...,Risk factors at medical school for subsequent professi...,Risk factors at medical school for subsequent professi...
4,Prevalence of frequent headache in a population sample.,Demographic Risk Factors for Alcohol-Related Aggressio...,Self-harm in prisons in England and Wales: an epidemio...,"Prevalence, risk factors, and uptake of interventions ..."
5,✓ Bullying in school: are short pupils at risk? Question...,Self-harm in prisons in England and Wales: an epidemio...,Inheritance of coronary artery disease in men: an anal...,Tuberculosis mortality in England and Wales during 198...


## A rare name

The claim hinges on two gene names. BM25 puts the paper first; bge-base and the fine-tuned MiniLM return papers on related genes and miss it in the top five, and the hybrid recovers it at rank 3.

In [4]:
show_validation("456")

Genes regulated by Esrrb transcription factor are sensitive to Mbd3 function.
relevant: An Oct4-Centered Protein Interaction Network in Embryonic Stem Cells


,BM25 (stemmed),bge-base,hybrid BM25 + bge-base,MiniLM fine-tuned
1,✓ An Oct4-Centered Protein Interaction Network in Embryo...,Epigenetic regulation of miR-184 by MBD1 governs neura...,"COMMD proteins, a novel family of structural and funct...",Transcriptional Repression of Gata3 Is Essential for E...
2,The developmental potential of iPSCs is greatly influe...,"COMMD proteins, a novel family of structural and funct...",Signal transducer and activator of transcription 3 is ...,KDM5B regulates embryonic stem cell self-renewal and r...
3,The TOR signalling pathway controls nuclear localizati...,The Sp1-like protein BTEB3 inhibits transcription via ...,✓ An Oct4-Centered Protein Interaction Network in Embryo...,The Translation Initiation Factor 3f (eIF3f) Exhibits ...
4,RNA sequencing identifies dysregulation of the human p...,Epigenetic regulation of NANOG by miR-302 cluster-MBD2...,The multidrug ABC transporter BmrC/BmrD of Bacillus su...,Hotspot mutations in H3F3A and IDH1 define distinct ep...
5,Nuclear PTEN functions as an essential regulator of SR...,The multidrug ABC transporter BmrC/BmrD of Bacillus su...,The Sp1-like protein BTEB3 inhibits transcription via ...,The Histone H3 Lysine-27 Demethylase Jmjd3 Links Infla...


## A paper the fine-tuned model misses

BM25, bge-base and the hybrid rank the judged paper first; the fine-tuned MiniLM returns a prevalence review on anxiety disorders and ranks the paper 14th.

In [5]:
show_validation("930")

Patients with panic anxiety show decreased CSF levels of hypocretin.
relevant: A KEY ROLE FOR OREXIN IN PANIC ANXIETY


,BM25 (stemmed),bge-base,hybrid BM25 + bge-base,MiniLM fine-tuned
1,✓ A KEY ROLE FOR OREXIN IN PANIC ANXIETY,✓ A KEY ROLE FOR OREXIN IN PANIC ANXIETY,✓ A KEY ROLE FOR OREXIN IN PANIC ANXIETY,Global prevalence of anxiety disorders: a systematic r...
2,Comorbidity of depressive and anxiety disorders in chr...,A diffusion tensor magnetic resonance imaging study of...,Comorbidity of depressive and anxiety disorders in chr...,Leptin treatment ameliorates anxiety in ob/ob obese mice.
3,How do I perform a lumbar puncture and analyze the res...,Comorbidity of depressive and anxiety disorders in chr...,A diffusion tensor magnetic resonance imaging study of...,Comorbidity of depressive and anxiety disorders in chr...
4,Exosomal levels of miRNA-21 from cerebrospinal fluids ...,Chronic daily headache. A clinical and psychological p...,Chronic daily headache. A clinical and psychological p...,The relation between past exposure to fine particulate...
5,Chronic daily headache. A clinical and psychological p...,Global prevalence of anxiety disorders: a systematic r...,The relation between past exposure to fine particulate...,Psychological treatment of malignant vasovagal syncope...


## Free-text query

Any text works. There are no relevance judgements, so there is no ✓.

In [6]:
show("vitamin D supplementation lowers the risk of respiratory infection")

,BM25 (stemmed),bge-base,hybrid BM25 + bge-base,MiniLM fine-tuned
1,"Vitamin D: The ""sunshine"" vitamin.","Vitamin D: The ""sunshine"" vitamin.","Vitamin D: The ""sunshine"" vitamin.","Vitamin D: The ""sunshine"" vitamin."
2,Vitamin D and prevention of colorectal cancer.,"Mammographic density, plasma vitamin D levels and risk...",Vitamin D and prevention of colorectal cancer.,Estimation of the dietary requirement for vitamin D in...
3,Vitamin D and obesity: current perspectives and future...,Vitamin D and obesity: current perspectives and future...,Vitamin D and obesity: current perspectives and future...,"Vitamin D status: measurement, interpretation, and cli..."
4,Estimation of the dietary requirement for vitamin D in...,Plasma vitamin D metabolites and risk of colorectal ca...,"Mammographic density, plasma vitamin D levels and risk...",[Vitamin D and latitude as environmental factors in mu...
5,Folate and vitamin B6 from diet and supplements in rel...,Environmental factors that influence the cutaneous pro...,Estimation of the dietary requirement for vitamin D in...,"Mammographic density, plasma vitamin D levels and risk..."


## Test-set results

nDCG@10 on the 300 test queries, scored once (`src/score_test.py`), 95% bootstrap interval over queries; fine-tuned rows are the three training seeds.

In [7]:
test = pd.read_csv("data/test_per_query.csv")
order = ["stemmed BM25", "MiniLM pretrained", "bge-base pretrained", "hybrid BM25 + bge-base"] + sorted(s for s in test["system"].unique() if "fine-tuned" in s)
rows = []
for name in order:
    part = test[test["system"] == name]
    mean, lo, hi = bootstrap_ci(part["ndcg@10"])
    rows.append({"system": name, "nDCG@10": f"{mean:.3f} ({lo:.3f} to {hi:.3f})", "recall@10": round(part["recall@10"].mean(), 3), "recall@100": round(part["recall@100"].mean(), 3)})
pd.DataFrame(rows).set_index("system")

,nDCG@10,recall@10,recall@100
system,,,
stemmed BM25,0.686 (0.639 to 0.730),0.808,0.916
MiniLM pretrained,0.645 (0.599 to 0.689),0.783,0.925
bge-base pretrained,0.740 (0.700 to 0.779),0.874,0.967
hybrid BM25 + bge-base,0.752 (0.711 to 0.791),0.875,0.980
"MiniLM fine-tuned, hard negatives, seed 0",0.668 (0.623 to 0.712),0.809,0.930
"MiniLM fine-tuned, hard negatives, seed 1",0.662 (0.617 to 0.707),0.809,0.927
"MiniLM fine-tuned, hard negatives, seed 2",0.652 (0.607 to 0.696),0.795,0.927
